In [ ]:
# Setup and Imports
import pandas as pd
import numpy as np
import kagglehub
import os
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import tukey_hsd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load datasets
urban_path = kagglehub.dataset_download("vellis1/us-cities-urban-connectivity")
yelp_path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

# Load urban connectivity data
files = os.listdir(urban_path)
csv_files = [f for f in files if f.endswith('.csv')]
urban_df = pd.read_csv(os.path.join(urban_path, csv_files[0]))

# Load Yelp business data
yelp_business_path = os.path.join(yelp_path, "yelp_academic_dataset_business.json")
yelp_df = pd.read_json(yelp_business_path, lines=True)

print(f"Urban data shape: {urban_df.shape}")
print(f"Yelp data shape: {yelp_df.shape}")

In [ ]:
# Category mapping rules - comprehensive keyword matching
MAJOR_RULES = {
    "Food & Dining": [
        "restaurant","food","cafe","coffee","tea","bar","bakery","dessert","ice cream",
        "pizza","burger","sandwich","noodle","ramen","sushi","mexican","italian","korean",
        "thai","indian","chinese","japanese","mediterranean","halal","vegan","vegetarian",
        "brunch","breakfast","deli","seafood","steakhouse","bbq","brew","wine","pub",
        "grocery","market","butcher","donut","bagel","taco","poke","buffet","food truck",
        "american (traditional)","american (new)","salad","chicken wings","cajun","creole",
        "southern","diners","latin american","soup","vietnamese","asian fusion",
        "middle eastern","gluten-free","tex-mex","greek","french","cuban","fish & chips",
        "dim sum","creperies","hawaiian","falafel","african","puerto rican","turkish",
        "german","cantonese","hot pot","filipino","brazilian","ethiopian","kosher",
        "pakistani","irish","szechuan","lebanese","kebab","brasseries","pan asian",
        "empanadas","persian","iranian","teppanyaki","malaysian","fondue","izakaya",
        "shanghainese","mongolian","himalayan","nepalese","ukrainian","egyptian",
        "singaporean","burmese","armenian","syrian","scandinavian","australian",
        "bangladeshi","sicilian","senegalese","haitian","trinidadian","iberian",
        "hungarian","somali","sardinian","georgian","sri lankan","guamanian",
        "serbo croatian","czech","hainan","israeli","fuzhou","south african",
        "colombian","peruvian","venezuelan","salvadoran","dominican","russian",
        "honduran","laotian","argentine","belgian","portuguese","british","moroccan",
        "afghan","arabic","polish","cambodian","indonesian","basque","taiwanese",
        "smokehouse","pretzels","waffles","shaved ice","acai bowls","gelato",
        "custom cakes","cupcakes","fruits & veggies","wraps","kombucha",
        "herbs & spices","olive oil","honey","pancakes","beer hall","poutineries",
        "bistros","street vendors","personal chefs","distilleries","cideries",
        "meaderies","speakeasies","beer","cucina campana","tuscan","austrian", "bakeries"
    ],
    "Shopping & Retail": [
        "store","shop","shopping","fashion","clothing","jewelry","gift","book","electronics",
        "furniture","thrift","antique","cosmetic","supply","wholesale","mall","marketplace",
        "shoe","accessories","toy","department","boutique",
        "home decor","sporting goods","mags","used","vintage & consignment",
        "mobile phones","building supplies","kitchen & bath","eyewear & opticians",
        "mattresses","watches","lighting fixtures","outdoor gear","guns & ammo",
        "leather goods","vinyl records","lingerie","costumes","formal wear",
        "motorcycle gear","hats","perfume","luggage","swimwear","knitting supplies",
        "customized merchandise","wigs","religious items","uniforms","kitchen supplies",
        "office equipment","rugs","shades & blinds","tableware","military surplus",
        "hunting & fishing supplies","gemstones & minerals","gold buyers",
        "diamond buyers","firewood","safety equipment","vitamins & supplements",
        "computers","signmaking","framing","grilling equipment"
    ],
    "Beauty & Personal Care": [
        "salon","spa","hair","nail","wax","tattoo","piercing","massage","skin","makeup",
        "barber","tanning","eyelash","threading","facial",
        "blow dry/out services","eyebrow services","sugaring","acne treatment",
        "aestheticians","estheticians"
    ],
    "Health & Medical": [
        "doctor","dentist","hospital","medical","clinic","therapy","chiropractor",
        "optometrist","psychologist","psychiatrist","surgeon","pharmacy","rehab",
        "urgent care","health","wellness","nutrition",
        "orthodontists","endodontists","periodontists","family practice",
        "acupuncture","reflexology","weight loss centers","diagnostic services",
        "dermatologists","ophthalmologists","internal medicine","obstetricians",
        "gynecologists","pediatricians","orthopedists","naturopathic","holistic",
        "reiki","laboratory testing","pain management","laser eye surgery","lasik",
        "allergists","gastroenterologist","neurologist","podiatrists","ear nose & throat",
        "radiologists","urologists","fertility","endocrinologists","oncologist",
        "sleep specialists","osteopathic physicians","audiologist","prosthodontists",
        "nurse practitioner","addiction medicine","dental hygienists","dietitians",
        "hearing aid providers","pulmonologist","speech therapists","vascular medicine",
        "emergency rooms","emergency medicine","iv hydration","colonics",
        "alternative medicine","meditation centers","tui na","tai chi","qi gong",
        "ayurveda","preventive medicine","concierge medicine","lactation services",
        "midwives","hospice","skilled nursing","assisted living facilities",
        "diagnostic imaging","teeth whitening","orthotics"
    ],
    "Automotive": [
        "auto","car","vehicle","tire","oil change","repair","dealer","towing","smog",
        "body shop","transmission","glass","detailing",
        "gas stations","roadside assistance","ev charging stations"
    ],
    "Home & Local Services": [
        "plumbing","electrician","hvac","roof","contractor","cleaning","locksmith",
        "moving","landscaping","flooring","installation","repair","maintenance","handyman",
        "appliance","home service","garden",
        "local services","movers","sewing & alterations","printing services",
        "shipping centers","pest control","painters","tree services",
        "junk removal & hauling","damage restoration","masonry/concrete",
        "packing services","pool & hot tub service","irrigation","window washing",
        "landscape architects","pool cleaners","gutter services","pressure washers",
        "garage door services","fences & gates","home window tinting","cabinetry",
        "tiling","laundry services","laundromat","refinishing services",
        "grout services","demolition services","stucco services","snow removal",
        "siding","decks & railing","chimney sweeps","fireplace services",
        "hydro-jetting","septic services","tv mounting","shutters","patio coverings",
        "home inspectors","home organization","home staging","awnings",
        "fire protection services","environmental abatement","screen printing",
        "interior design","florists","floral designers","swimming pools",
        "hot tub & pool","home energy auditors","well drilling","excavation services",
        "backflow services","wallpapering","powder coating","grill services",
        "wildlife control","home developers","interlock systems",
        "sheds & outdoor storage","security systems","security services",
        "environmental testing"
    ],
    "Real Estate & Housing": [
        "real estate","apartment","property","mortgage","housing","leasing",
        "condominiums","self storage","homeowner association"
    ],
    "Professional & Financial Services": [
        "law","legal","account","insurance","tax","consult","financial","bank",
        "notary","payday","broker","business service",
        "professional services","general litigation","investing",
        "check cashing/pay-day loans","title loans","installment loans",
        "debt relief services","payroll services","employment agencies",
        "advertising","graphic design","web design","appraisal services",
        "registration services","personal assistants","wills","trusts","& probates",
        "business financing","process servers","editorial services",
        "translation services","software development","billing services",
        "data recovery","internet service providers","utilities",
        "electricity suppliers","talent agencies","structural engineers"
    ],
    "Entertainment & Recreation": [
        "gym","fitness","park","museum","art","music","theater","cinema","club",
        "karaoke","festival","sports","golf","yoga","dance","arcade","game",
        "recreation","bowling","casino",
        "nightlife","active life","event planning & services","lounges",
        "trainers","boot camps","pilates","jazz & blues","kids activities",
        "pool halls","boxing","stadiums & arenas","skating rinks","hiking",
        "boating","kickboxing","tennis","mountain biking","climbing",
        "brazilian jiu-jitsu","horseback riding","fishing","gun/rifle ranges",
        "rafting/kayaking","axe throwing","paint & sip","challenge courses",
        "diving","scuba diving","rock climbing","laser tag","paintball",
        "virtual reality centers","archery","skydiving","surfing","tubing",
        "hot air balloons","batting cages","sailing","free diving",
        "adult entertainment","adult","haunted houses","scavenger hunts",
        "jet skis","soccer","basketball courts","baseball fields","marinas",
        "opera & ballet","playgrounds","summer camps","day camps",
        "indoor playcentre","aquariums","zoos","landmarks & historical buildings",
        "local flavor","libraries","beaches","lakes","campgrounds",
        "horse racing","race tracks","djs","pool & billiards","eatertainment",
        "pumpkin patches","attraction farms","bike sharing","bicycle paths",
        "pickleball","bocce ball","badminton","skiing","sledding",
        "outdoor movies","bingo halls","rodeo","airsoft","cannabis dispensaries",
        "cannabis collective","bikes","swimming pools","snowboarding",
        "hang gliding","ziplining","snorkeling","trivia hosts",
        "recording & rehearsal studios","video/film production","magicians",
        "face painting","clowns","wildlife hunting ranges","metal detector services"
    ],
    "Education": [
        "school","college","university","tutor","training","education","class","lesson",
        "test preparation"
    ],
    "Travel & Transportation": [
        "hotel","travel","tour","airport","taxi","transport","rental","shuttle",
        "airline","resort","vacation",
        "limos","guest houses","hostels","train stations","buses","bus stations",
        "ferries","pedicabs","valet services","luggage storage","rest stops",
        "visitor centers","trains","flight instruction"
    ],
    "Pets": [
        "pet","veterinarian","animal","dog","cat","groomer","boarding",
        "aquarium services"
    ],
    "Community & Public Services": [
        "community service/non-profit","religious organizations","churches",
        "cultural center","community centers","funeral services & cemeteries",
        "cremation services","mortuary services","post offices","courthouses",
        "donation center","recycling center","blood & plasma donation centers",
        "synagogues","buddhist temples","hindu temples","mosques",
        "homeless shelters","jails & prisons","municipality","embassy",
        "crisis pregnancy centers","faith-based crisis pregnancy centers",
        "adoption services","hospice"
    ],
    "Events & Weddings": [
        "wedding planning","bridal","photographers","session photography",
        "event photography","boudoir photography","videographers",
        "wedding chapels","officiants","balloon services","holiday decorations",
        "holiday decorating services","christmas trees","paint-your-own pottery",
        "screen printing/t-shirt printing","engraving","calligraphy",
        "digitizing services","customized merchandise"
    ]
}

def map_major_category(category):
    if pd.isna(category):
        return "Other"
    c = category.lower()
    for major, keywords in MAJOR_RULES.items():
        for k in keywords:
            if k in c:
                return major
    return "Other"

In [ ]:
# Data Cleaning and Merging
urban_df = urban_df.rename(columns={"State": "state", "City": "city"})

# Merge on both city and state for accurate geographic matching
combined = pd.merge(left=urban_df, right=yelp_df, 
                    left_on=["city", "state"], 
                    right_on=["city", "state"], 
                    how="inner")

# Process categories
combined['categories'] = combined['categories'].str.split(", ")
combined = combined.explode('categories')
combined["major_category"] = combined["categories"].apply(map_major_category)

# Create Travel Score (avg of all scores)
combined['Travel_Score'] = (combined['Walk Score'] + combined['Transit Score'] + combined['Bike Score']) / 3

# Filter to open businesses only
combined = combined[combined['is_open'] == 1]

# Keep relevant columns
keep_cols = ['Place_name', 'city', 'state', 'Walk Score', 'Transit Score', 'Bike Score',
            'business_id', 'name', 'postal_code', 'latitude', 'longitude', 
            'stars', 'review_count', 'major_category', 'Travel_Score']
combined = combined[[col for col in keep_cols if col in combined.columns]]

# Deduplicate by business_id
combined_deduped = combined.drop_duplicates(subset=['business_id'])

print(f"Combined dataset shape: {combined.shape}")
print(f"Unique businesses: {combined_deduped.shape[0]}")
print(f"Unique cities: {combined_deduped['city'].nunique()}")

In [ ]:
# Create multi-hot encoded dataset for multivariate analysis
combined_onehot = pd.get_dummies(combined[['business_id', 'major_category']], 
                                columns=['major_category'], dtype=int)

# Group by business_id to create multi-hot encoding
category_cols = [col for col in combined_onehot.columns if col.startswith('major_category_')]
business_categories = combined_onehot.groupby('business_id')[category_cols].max().reset_index()

# Merge back with business data
combined_final = combined_deduped.drop(columns=['major_category']).merge(business_categories, on='business_id')
combined_final = combined_final.dropna()

print(f"Final dataset shape: {combined_final.shape}")